# 04 ML Baseline Review

This notebook checks whether classical baselines beat dummy models and whether each modality is credible before CNN or fusion work.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/home/ubuntu/nishn_workspce/test_pdfs_generic/.covid_audio_btp_private/covid_audio_btp")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from covid_audio_btp.notebook_utils import (
    PROJECT_ROOT,
    artifact,
    check_artifacts,
    count_table,
    read_csv,
    read_optional_csv,
    require_artifacts,
    save_table,
    value_counts_frame,
    assert_no_participant_leakage,
    assert_binary_labels_present,
    stop_if_validation_errors,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
print(PROJECT_ROOT)


In [ ]:
require_artifacts([
    "data/outputs/metrics/ml_baseline_metrics.csv",
    "data/outputs/metrics/ml_predictions_validation.csv",
    "data/outputs/metrics/ml_predictions_test.csv",
])
metrics = read_csv("data/outputs/metrics/ml_baseline_metrics.csv")
val_pred = read_csv("data/outputs/metrics/ml_predictions_validation.csv", n=5)
test_pred = read_csv("data/outputs/metrics/ml_predictions_test.csv", n=5)


## Ranking And Dummy Baseline Gate


In [ ]:
metric_cols = [c for c in ["auroc", "auprc", "balanced_accuracy", "f1", "sensitivity", "specificity", "brier", "ece", "nll", "n_samples"] if c in metrics.columns]
ranking = metrics.sort_values(["modality", "auprc"], ascending=[True, False])[["model_name", "modality"] + metric_cols]
save_table(ranking, "reports/tables/nb04_ml_baseline_ranking.csv")
display(ranking)

dummy = metrics[metrics["model_name"].str.startswith("dummy", na=False)].groupby("modality")["auprc"].max()
real = metrics[~metrics["model_name"].str.startswith("dummy", na=False)].groupby("modality")["auprc"].max()
comparison = pd.concat([dummy.rename("best_dummy_auprc"), real.rename("best_real_auprc")], axis=1)
comparison["margin"] = comparison["best_real_auprc"] - comparison["best_dummy_auprc"]
save_table(comparison.reset_index(), "reports/tables/nb04_dummy_vs_real.csv")
display(comparison)
if (comparison["margin"].dropna() <= 0).any():
    print("Warning: at least one modality does not beat dummy baseline. Do not overclaim.")


## Metric Plots


In [ ]:
plot_metrics = [c for c in ["auroc", "auprc", "ece", "brier"] if c in metrics.columns]
fig, axes = plt.subplots(1, len(plot_metrics), figsize=(5 * len(plot_metrics), 4))
if len(plot_metrics) == 1:
    axes = [axes]
for ax, col in zip(axes, plot_metrics):
    sns.barplot(data=metrics, x="modality", y=col, hue="model_name", ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "reports/figures/nb04_ml_metrics.png", dpi=160)
plt.show()


## Probability Sanity Checks


In [ ]:
for name, pred in [("validation", val_pred), ("test", test_pred)]:
    print(name)
    display(pred.groupby(["model_name", "modality"])["probability"].describe().reset_index().head(30))
    out_of_range = pred[~pred["probability"].between(0, 1)]
    if len(out_of_range):
        raise AssertionError(f"{name} predictions contain probabilities outside [0, 1]")
